In [ ]:
import json
import csv
import pandas as pd
from pathlib import Path
from tqdm.cli import tqdm
from rapidfuzz import fuzz

In [ ]:
from sys import path as sys_path
project_root = Path.cwd().parent.parent
if project_root not in sys_path:
    sys_path.append(str(project_root))

In [ ]:
from src.duplicate_page_discovery import DuplicateConfig, process_parish

In [ ]:
data_dir = project_root/"data/"
outputs_dir = project_root/"outputs/"

## Load used data

Migration record formats metadata

In [ ]:
book_migration_directions = pd.read_csv(
    outputs_dir/"csv/book_migration_direction_info.csv"
)

In [ ]:
with open(outputs_dir/"json/book_ids.json", "r") as fp:
    book_id_to_info = json.load(fp)

## Duplicate filtering

### Deciding on similarity metric

In [ ]:
with open(
    data_dir/"migration-data-csv-release-v1/tampereen_tuomiokirkkoseurakunta/"
    "muuttaneet_1898-1903_ap/"
    "autods_tampereen_tuomiokirkkoseurakunta_muuttaneet_1898-1903_ap_5_left_5.csv",
    mode="r"
) as fp:
    page_a = "|".join([",".join(row) for row in csv.reader(fp)])

with open(
    data_dir/"migration-data-csv-release-v1/tampereen_tuomiokirkkoseurakunta/"
    "muuttaneet_1898-1903_mko40-50/"
    "autods_tampereen_tuomiokirkkoseurakunta_muuttaneet_1898-1903_mko40-50_1_left_5.csv",
    mode="r"
) as fp:
    page_b = "|".join([",".join(row) for row in csv.reader(fp)])


with open(
    data_dir/"migration-data-csv-release-v1/tampereen_tuomiokirkkoseurakunta/"
    "muuttaneet_1898-1903_mko40-50/"
    "autods_tampereen_tuomiokirkkoseurakunta_muuttaneet_1898-1903_mko40-50_2_left_5.csv",
    mode="r"
) as fp:
    page_c = "|".join([",".join(row) for row in csv.reader(fp)])

with open(
    data_dir/"migration-data-csv-release-v1/turun_tuomiokirkkosrk/"
    "muuttaneet_1892-1898_ap/"
    "autods_turun_tuomiokirkkosrk_muuttaneet_1892-1898_ap_5_left_5.csv",
    mode="r"
) as fp:
    page_d = "|".join([",".join(row) for row in csv.reader(fp)])

with open(
    data_dir/"migration-data-csv-release-v1/turun_ruotsalainen/"
    "muuttaneet_1825-1835_ap/"
    "autods_turun_ruotsalainen_muuttaneet_1825-1835_ap_4_left_5.csv",
    mode="r"
) as fp:
    page_e = "|".join([",".join(row) for row in csv.reader(fp)])

with open(
    data_dir/"migration-data-csv-release-v1/turun_ruotsalainen/"
    "muuttaneet_1825-1835_jk1126/"
    "autods_turun_ruotsalainen_muuttaneet_1825-1835_jk1126_6_left_5.csv",
    mode="r"
) as fp:
    page_f = "|".join([",".join(row) for row in csv.reader(fp)])

with open(
    data_dir/"migration-data-csv-release-v1/ylojarvi/"
    "muuttaneet_1844-1884_tk225/"
    "autods_ylojarvi_muuttaneet_1844-1884_tk225_6_left_5.csv",
    mode="r"
) as fp:
    page_g = "|".join([",".join(row) for row in csv.reader(fp)])
with open(
    data_dir/"migration-data-csv-release-v1/ylojarvi/"
    "muuttaneet_1844-1890_ap/"
    "autods_ylojarvi_muuttaneet_1844-1890_ap_4_left_5.csv",
    mode="r"
) as fp:
    page_h = "|".join([",".join(row) for row in csv.reader(fp)])

In [83]:
results = {
    "Same book, same page (1/3)": {
        "fuzz.ratio": fuzz.ratio(page_a, page_b),
        "fuzz.WRatio": fuzz.WRatio(page_a, page_b),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_a, page_b),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_a, page_b),
    },
    "Same book, same page (2/3)": {
        "fuzz.ratio": fuzz.ratio(page_e, page_f),
        "fuzz.WRatio": fuzz.WRatio(page_e, page_f),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_e, page_f),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_e, page_f),
    },
    "Same book, same page (3/3)": {
        "fuzz.ratio": fuzz.ratio(page_g, page_h),
        "fuzz.WRatio": fuzz.WRatio(page_g, page_h),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_g, page_h),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_g, page_h),
    },
    "Same book, different page": {
        "fuzz.ratio": fuzz.ratio(page_a, page_c),
        "fuzz.WRatio": fuzz.WRatio(page_a, page_c),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_a, page_c),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_a, page_c),
    },
    "Different book, different page (1/3)": {
        "fuzz.ratio": fuzz.ratio(page_a, page_d),
        "fuzz.WRatio": fuzz.WRatio(page_a, page_d),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_a, page_d),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_a, page_d),
    },
    "Different book, different page (2/3)": {
        "fuzz.ratio": fuzz.ratio(page_b, page_d),
        "fuzz.WRatio": fuzz.WRatio(page_b, page_d),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_b, page_d),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_b, page_d),
    },
    "Different book, different page (3/3)": {
        "fuzz.ratio": fuzz.ratio(page_c, page_d),
        "fuzz.WRatio": fuzz.WRatio(page_c, page_d),
        "fuzz.token_sort_ratio": fuzz.token_sort_ratio(page_c, page_d),
        "fuzz.token_set_ratio": fuzz.token_set_ratio(page_c, page_d),
    }
}
pd.DataFrame.from_dict(results, orient="index")

,fuzz.ratio,fuzz.WRatio,fuzz.token_sort_ratio,fuzz.token_set_ratio
"Same book, same page (1/3)",79.787503,79.787503,62.531646,63.434022
"Same book, same page (2/3)",82.037815,82.037815,67.577509,70.314193
"Same book, same page (3/3)",76.392252,76.392252,55.316571,56.426332
"Same book, different page",42.785571,44.102698,44.622712,46.423893
"Different book, different page (1/3)",35.345035,35.345035,36.814244,36.752806
"Different book, different page (2/3)",34.576271,35.272154,36.724806,37.128583
"Different book, different page (3/3)",34.252818,34.252818,34.029278,34.945398


### Running duplicate page discovery

In [ ]:
jsonls = tuple(
    Path(outputs_dir/"json/all_migration_events_parish_splits").glob("*.jsonl")
)
jsonls_tampere = tuple(
    filter(lambda x: "543" in x.stem, jsonls)
)

In [ ]:
config = DuplicateConfig(
    scorer=fuzz.ratio,
    threshold=75,
    max_year_diff=1,
    correct_year_prediction_bounds=(1500, 1980),
    join_char="|"
)

output_dir = outputs_dir/"json/all_migration_events_parish_splits_duplicate_info/"
for parish_jsonl in tqdm(jsonls):
    process_parish(parish_jsonl, output_dir, config)

100%|██████████| 468/468 [14:38<00:00,  1.88s/it]
